# Task
Analyze and visualize copy ratio profiles from "denoised_copy_ratios.tsv" and "segments.tsv" files. Generate individual plots for each chromosome and a combined plot for all chromosomes, displaying genomic positions in Mb on the x-axis.

## Identify chromosomes

### Subtask:
Get the list of unique chromosomes present in the data.


**Reasoning**:
Extract the unique values from the 'CONTIG' column of the `denoised` DataFrame and store them in a list called `chromosomes`.



## Plot individual chromosomes

### Subtask:
Iterate through the list of chromosomes and generate a separate plot for each one, similar to the current chr8 plot.


**Reasoning**:
Iterate through the list of chromosomes and generate a separate plot for each one, similar to the current chr8 plot.



## Plot all chromosomes

### Subtask:
Generate a single plot containing all chromosomes. This might require adjusting the x-axis to represent the entire genome.


**Reasoning**:
Calculate the cumulative genomic positions for both denoised and segments dataframes and then create the genome-wide scatter plot with segment lines and chromosome boundaries.



## Summary:

### Data Analysis Key Findings

* The data contains 27 unique chromosomes, including standard chromosomes (chr1-chr22, chrX, chrY), mitochondrial DNA (chrM), and some alternative contigs.
* Individual plots were successfully generated for each chromosome, displaying the denoised copy ratio data as scatter plots and overlaying the segment means as horizontal lines.
* A genome-wide plot was created by calculating cumulative genomic positions across all chromosomes, allowing the visualization of denoised copy ratios and segment means on a single plot with chromosome boundaries indicated.

### Insights or Next Steps

* The plots effectively visualize copy number alterations across the genome, highlighting regions of potential gains or losses as indicated by the segment mean log2 copy ratios deviating from zero.
* Further analysis could involve identifying specific genes located within aberrant segments and correlating these findings with other genomic or clinical data.

**Reasoning**:
Generate a cell with code to save the plots. This code can be adapted for both individual and genome-wide plots.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

# === Directory containing your files ===
data_dir = "file/"
output_dir = "plots/"
os.makedirs(output_dir, exist_ok=True)

# === Find all denoised and segment files ===
denoised_files = sorted(glob.glob(os.path.join(data_dir, "*.denoisedCR.tsv")))
segment_files = sorted(glob.glob(os.path.join(data_dir, "*.called.seg")))

# === Helper: chromosome sorting ===
def sort_chromosomes(chr_name):
    if chr_name.startswith('chr'):
        suffix = chr_name[3:]
        if suffix.isdigit():
            return int(suffix)
        elif suffix == 'X':
            return 23
        elif suffix == 'Y':
            return 24
    return 25  # unexpected contigs

# === Loop through all samples ===
for denoised_file in denoised_files:
    sample_id = os.path.basename(denoised_file).split(".")[0]
    seg_file = denoised_file.replace(".denoisedCR.tsv", ".called.seg")

    if not os.path.exists(seg_file):
        print(f"⚠️ Skipping {sample_id} (no segment file found)")
        continue

    print(f"📊 Plotting {sample_id} ...")

    # --- Load data ---
    denoised = pd.read_csv(denoised_file, sep='\t', comment='@')
    segments = pd.read_csv(seg_file, sep='\t', comment='@')

    # --- Get chromosomes ---
    chromosomes = denoised['CONTIG'].unique().tolist()
    chromosomes = [c for c in chromosomes if c.startswith('chr') and '_alt' not in c and c != 'chrM']
    chromosomes.sort(key=sort_chromosomes)

    # --- Plot per chromosome ---
    for chromosome in chromosomes:
        denoised_chr = denoised[denoised['CONTIG'] == chromosome]
        segments_chr = segments[segments['CONTIG'] == chromosome]

        plt.figure(figsize=(12, 5))
        plt.scatter(denoised_chr['START'], denoised_chr['LOG2_COPY_RATIO'], color='grey', s=0.5, label='Denoised')

        for _, row in segments_chr.iterrows():
            plt.plot([row['START'], row['END']],
                     [row['MEAN_LOG2_COPY_RATIO'], row['MEAN_LOG2_COPY_RATIO']],
                     color='red', linewidth=2)

        plt.title(f'{sample_id} - {chromosome}')
        plt.xlabel('Genomic Position (Mb)')
        plt.ylabel('log2 Copy Ratio')
        plt.ylim(-4, plt.ylim()[1])
        plt.tight_layout()
        plt.savefig(f"{output_dir}/{sample_id}_{chromosome}.png", dpi=150)
        plt.close()

    # --- Compute chromosome offsets for genome-wide plot ---
    chromosome_sizes = denoised[denoised['CONTIG'].isin(chromosomes)].groupby('CONTIG')['END'].max()
    chromosome_offsets = np.cumsum([0] + [chromosome_sizes[c] for c in chromosomes[:-1]])
    chromosome_offset_map = {contig: offset for contig, offset in zip(chromosomes, chromosome_offsets)}

    denoised['CUMULATIVE_START'] = denoised.apply(lambda row: row['START'] + chromosome_offset_map.get(row['CONTIG'], 0), axis=1)
    denoised['CUMULATIVE_END'] = denoised.apply(lambda row: row['END'] + chromosome_offset_map.get(row['CONTIG'], 0), axis=1)
    segments['CUMULATIVE_START'] = segments.apply(lambda row: row['START'] + chromosome_offset_map.get(row['CONTIG'], 0), axis=1)
    segments['CUMULATIVE_END'] = segments.apply(lambda row: row['END'] + chromosome_offset_map.get(row['CONTIG'], 0), axis=1)

    # --- Plot genome-wide view ---
    plt.figure(figsize=(20, 6))
    plt.scatter(denoised['CUMULATIVE_START'], denoised['LOG2_COPY_RATIO'], color='grey', s=0.5, alpha=0.5, label='Denoised')

    for _, row in segments.iterrows():
        plt.plot([row['CUMULATIVE_START'], row['CUMULATIVE_END']],
                 [row['MEAN_LOG2_COPY_RATIO'], row['MEAN_LOG2_COPY_RATIO']],
                 color='red', linewidth=2)

    # Add chromosome boundaries and labels
    for i, contig in enumerate(chromosomes):
        offset = chromosome_offset_map[contig]
        plt.axvline(x=offset, color='blue', linestyle='--', linewidth=0.5, alpha=0.5)

    chromosome_centers = [chromosome_offset_map[c] + chromosome_sizes[c]/2 for c in chromosomes]
    plt.xticks(chromosome_centers, chromosomes, rotation=90)
    plt.title(f'Genome-wide Copy Ratio Profile - {sample_id}')
    plt.xlabel('Chromosomes')
    plt.ylabel('log2 Copy Ratio')
    plt.ylim(-4, plt.ylim()[1])
    plt.tight_layout()
    plt.savefig(f"{output_dir}/{sample_id}_genomewide.png", dpi=150)
    plt.close()

print("✅ All plots generated successfully!")


📊 Plotting WIAB_IDPE_10 ...
📊 Plotting WIAB_IDPE_14 ...
📊 Plotting WIAB_IDPE_16 ...
📊 Plotting WIAB_IDPE_2 ...
📊 Plotting WIAB_IDPE_20 ...
📊 Plotting WIAB_IDPE_24 ...
📊 Plotting WIAB_IDPE_26 ...
📊 Plotting WIAB_IDPE_28 ...
📊 Plotting WIAB_IDPE_30 ...
📊 Plotting WIAB_IDPE_32 ...
📊 Plotting WIAB_IDPE_4 ...
📊 Plotting WIAB_IDPE_8 ...
✅ All plots generated successfully!
